<a href="https://colab.research.google.com/github/basilhakim/ALPR-MINI-PROJECT/blob/main/NLPgloveNrnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 3: Sentence Completion with GloVe and RNN
(2215315)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# Libraries for downloading GloVe
import requests
import zipfile
import os

In [ ]:
# 1. Define the toy corpus [cite: 5, 6]
corpus = [
    "I love natural language processing",
    "Deep learning models can learn patterns",
    "Recurrent neural networks are useful",
    "Word embeddings capture semantic meaning",
    "LSTM networks remember long sequences"
]

# 2. Tokenize the corpus [cite: 36]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1  # Add 1 for OOV (out-of-vocabulary) words

# 3. Create input sequences [cite: 39]
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 4. Pad sequences [cite: 46]
max_seq_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

# 5. Split predictors (X) and label (y) [cite: 50, 53]
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# 6. One-hot encode the label (y) [cite: 54]
y = to_categorical(y, num_classes=total_words)

In [ ]:
# 1. Download GloVe embeddings [cite: 56]
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
glove_zip_file = "glove.6B.zip"
glove_file = "glove.6B.50d.txt" # As specified: 50-dimensional

if not os.path.exists(glove_zip_file):
    print("Downloading GloVe embeddings...")
    r = requests.get(glove_url)
    with open(glove_zip_file, 'wb') as f:
        f.write(r.content)
    print("Download complete.")

# 2. Unzip the file
if not os.path.exists(glove_file):
    print("Unzipping GloVe embeddings...")
    with zipfile.ZipFile(glove_zip_file, 'r') as zip_ref:
        zip_ref.extract(glove_file)
    print("Unzip complete.")

# 3. Load embeddings into a dictionary
print("Loading embeddings into memory...")
embeddings_index = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
print(f"Found {len(embeddings_index)} word vectors.")

Download complete.
Unzipping GloVe embeddings...
Unzip complete.
Loading embeddings into memory...
Found 400000 word vectors.


In [ ]:
# 1. Define embedding dimension (must match GloVe file)
embedding_dim = 50

# 2. Create the embedding matrix
embedding_matrix = np.zeros((total_words, embedding_dim))

# 3. Populate the matrix
for word, i in tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

print("Embedding matrix prepared.")

Embedding matrix prepared.


In [ ]:
# Build the LSTM model [cite: 11, 60]
model = Sequential()

# Add the Embedding layer
model.add(Embedding(
    input_dim=total_words,                 # Size of the vocabulary [cite: 61]
    output_dim=embedding_dim,              # Dimension of embeddings (50) [cite: 63]
    weights=[embedding_matrix],            # Pre-trained weights [cite: 64]
    input_length=max_seq_len - 1,          # Length of input sequences [cite: 65]
    trainable=False                        # Freeze the embedding layer [cite: 66]
))

# Add the LSTM layer
model.add(LSTM(100))

# Add the output layer [cite: 67]
model.add(Dense(total_words, activation='softmax'))

# Compile the model [cite: 68]
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Print a summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │         1,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,300 (5.08 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,300 (5.08 KB)

In [ ]:
# Train your model using the sequences
print("Training the model...")
model.fit(X, y, epochs=100, verbose=1)
print("Training complete.")

Training the model...
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0000e+00 - loss: 3.2936
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.0952 - loss: 3.2503
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.0952 - loss: 3.2077
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.1905 - loss: 3.1657
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2381 - loss: 3.1241
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2857 - loss: 3.0829
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.2857 - loss: 3.0417
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.3333 - loss: 3.0005
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3810 - loss: 2.9590
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4286 - loss: 2.9171
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.4286 - loss: 2.8747
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

In [ ]:
# Implement a function to complete a sentence
def complete_sentence(seed_text, next_words=3):
    """
    Completes a sentence given a seed text.
    """
    for _ in range(next_words):
        # Tokenize the seed text
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Pad the sequence
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        # Predict the next word
        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted_index = np.argmax(predicted_probs)

        # Find the word corresponding to the index
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Append the predicted word to the seed text
        seed_text += " " + output_word

    return seed_text

In [ ]:
# Test your sentence completion
# Using "i love" as the seed (tokenizer lowercases "I love")
seed_1 = "i love"
print(f"Seed: '{seed_1}'")
print(f"Completion: '{complete_sentence(seed_1, next_words=3)}'")

print("---")

# Another test
seed_2 = "deep learning"
print(f"Seed: '{seed_2}'")
print(f"Completion: '{complete_sentence(seed_2, next_words=3)}'")

Seed: 'i love'
Completion: 'i love natural language processing'
---
Seed: 'deep learning'
Completion: 'deep learning models can learn'


**Summary of what we've done:**

1. *Data Preparation*: We defined a small corpus of sentences and then tokenized it to prepare the text data for the model.
2. *GloVe Embeddings*: We downloaded pre-trained GloVe word embeddings (50-dimensional) and loaded them into memory. These embeddings capture semantic relationships between words.
3. *Embedding Matrix Creation*: An embedding matrix was created, mapping the words in our corpus to their corresponding GloVe vectors. Words not found in GloVe were initialized with zeros.
4. *Model Building*: A recurrent neural network (RNN) model, specifically an LSTM (Long Short-Term Memory) network, was built using TensorFlow/Keras. This model includes:
An Embedding layer initialized with the pre-trained GloVe embedding matrix, which was set to be non-trainable (frozen).
An LSTM layer to learn sequential patterns in the text.
A Dense (output) layer with a softmax activation to predict the probability of the next word.
5. *Model Training*: The LSTM model was trained on the prepared input sequences (n-grams from the corpus) to learn to predict the next word in a sequence.
6. *Sentence Completion Function*: A function was implemented to take a seed text, tokenize it, use the trained model to predict subsequent words, and append them to complete the sentence.
7. *Testing*: Finally, we tested the sentence completion function with example seed texts like "i love" and "deep learning" to observe its predictions, and we successfully resolved a NameError that occurred during testing.